In [2]:
!pip install -q wfdb pandas numpy matplotlib scipy PyWavelets

import wfdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.signal import welch
import pywt

print("Libraries imported successfully!")

Matplotlib is building the font cache; this may take a moment.


Libraries imported successfully!


In [9]:
record_list = wfdb.get_record_list("tpehgdb")
print("Total recordings:", len(record_list))
print("\nFirst 10 recordings:")
print(record_list[:10])
print("\nLast 10 recordings:")
print(record_list[-10:])

Total recordings: 300

First 10 recordings:
['tpehgdb/tpehg1007', 'tpehgdb/tpehg1021', 'tpehgdb/tpehg1022', 'tpehgdb/tpehg1027', 'tpehgdb/tpehg1029', 'tpehgdb/tpehg1031', 'tpehgdb/tpehg1039', 'tpehgdb/tpehg1065', 'tpehgdb/tpehg1070', 'tpehgdb/tpehg1088']

Last 10 recordings:
['tpehgdb/tpehg938', 'tpehgdb/tpehg939', 'tpehgdb/tpehg947', 'tpehgdb/tpehg949', 'tpehgdb/tpehg961', 'tpehgdb/tpehg962', 'tpehgdb/tpehg966', 'tpehgdb/tpehg980', 'tpehgdb/tpehg991', 'tpehgdb/tpehg994']


In [11]:
test_record_name = record_list[0]
test_record = wfdb.rdrecord(test_record_name, pn_dir="tpehgdb/1.0.1/tpehgdb")
print("Record name:", test_record_name)
print("Sampling frequency:", test_record.fs)
print("Signal shape:", test_record.p_signal.shape)
print("Number of channels:", test_record.n_sig)


Record name: tpehgdb/tpehg1007
Sampling frequency: 20
Signal shape: (35180, 12)
Number of channels: 12


In [13]:
def extract_features(record_name):
    # 1. Load recording
    record_name = str(record_name).split("/")[-1]
    record = wfdb.rdrecord(record_name, pn_dir="tpehgdb/1.0.1/tpehgdb")
    fs = record.fs
    
    # 2. Select S1 (0.08–4 Hz)
    signal = record.p_signal[:, 1].astype(float)
    if not np.all(np.isfinite(signal)):
        raise ValueError("Signal contains missing or invalid values.")
        
    # 3. Remove first and last 180 seconds
    edge_samples = int(180 * fs)
    if len(signal) <= 2 * edge_samples:
        raise ValueError("Recording is too short after cropping.")
    clean_signal = signal[edge_samples : -edge_samples]
    
    # 4. Time-domain features
    mean_value = np.mean(clean_signal)
    variance = np.var(clean_signal)
    std_value = np.std(clean_signal)
    rms = np.sqrt(np.mean(clean_signal ** 2))
    peak_to_peak = np.max(clean_signal) - np.min(clean_signal)
    
    features = {
        'record_name': record_name,
        'mean': mean_value,
        'variance': variance,
        'std': std_value,
        'rms': rms,
        'peak_to_peak': peak_to_peak
    }
    return features

In [14]:
features = extract_features("tpehg1007")

In [15]:
from scipy.signal import welch, spectrogram
from scipy.integrate import cumulative_trapezoid

print("Additional functions imported successfully!")

Additional functions imported successfully!


In [16]:
def extract_features(record_name):

    # 1. Load recording
    record_name = str(record_name).split("/")[-1]

    record = wfdb.rdrecord(
        record_name,
        pn_dir="tpehgdb/1.0.1/tpehgdb"
    )

    fs = record.fs

    # 2. Select S1 (0.08–4 Hz)
    signal = record.p_signal[:, 1].astype(float)

    if not np.all(np.isfinite(signal)):
        raise ValueError("Signal contains missing or invalid values.")

    # 3. Remove first and last 180 seconds
    edge_samples = int(180 * fs)

    if len(signal) <= 2 * edge_samples:
        raise ValueError("Recording is too short after cropping.")

    clean_signal = signal[edge_samples:-edge_samples]

    # 4. Time-domain features
    mean_value = np.mean(clean_signal)
    variance = np.var(clean_signal)
    std_value = np.std(clean_signal)

    rms = np.sqrt(np.mean(clean_signal ** 2))

    peak_to_peak = (
        np.max(clean_signal) - np.min(clean_signal)
    )

    zero_crossings = np.sum(
        clean_signal[:-1] * clean_signal[1:] < 0
    )

    zcr = zero_crossings / (len(clean_signal) - 1)

    waveform_length = np.sum(
        np.abs(np.diff(clean_signal))
    )

    # 5. Frequency-domain features using Welch PSD
    frequencies, psd = welch(
        clean_signal,
        fs=fs,
        window="hann",
        nperseg=2048,
        noverlap=1024,
        scaling="density"
    )

    band_mask = (
        (frequencies >= 0.08) &
        (frequencies <= 4.0)
    )

    band_frequencies = frequencies[band_mask]
    band_psd = psd[band_mask]

    total_power = np.trapezoid(
        band_psd,
        band_frequencies
    )

    if total_power <= 0:
        raise ValueError("Invalid or zero band power.")

    mean_frequency = (
        np.trapezoid(
            band_frequencies * band_psd,
            band_frequencies
        ) / total_power
    )

    cumulative_power = cumulative_trapezoid(
        band_psd,
        band_frequencies,
        initial=0
    )

    median_frequency = np.interp(
        total_power / 2,
        cumulative_power,
        band_frequencies
    )

    peak_frequency = band_frequencies[
        np.argmax(band_psd)
    ]

    # 6. Select first five minutes for time-frequency analysis
    tf_duration = 300
    tf_samples = int(tf_duration * fs)

    if len(clean_signal) < tf_samples:
        raise ValueError("Not enough clean signal for 5-minute analysis.")

    segment = clean_signal[:tf_samples]

    # 7. STFT / Spectrogram
    stft_frequencies, stft_times, Sxx = spectrogram(
        segment,
        fs=fs,
        window="hann",
        nperseg=512,
        noverlap=384,
        scaling="density",
        mode="psd"
    )

    stft_mask = (
        (stft_frequencies >= 0.08) &
        (stft_frequencies <= 4.0)
    )

    stft_band = Sxx[stft_mask, :]

    stft_band_frequencies = stft_frequencies[stft_mask]

    mean_spectrogram_power = np.mean(stft_band)

    max_spectrogram_power = np.max(stft_band)

    stft_average_power = np.mean(stft_band, axis=1)

    dominant_stft_frequency = stft_band_frequencies[
        np.argmax(stft_average_power)
    ]

    # 8. Continuous Wavelet Transform (CWT)
    wavelet = "cmor1.5-1.0"

    target_frequencies = np.linspace(0.08, 4.0, 80)

    central_frequency = pywt.central_frequency(wavelet)

    scales = (
        central_frequency * fs / target_frequencies
    )

    coefficients, actual_frequencies = pywt.cwt(
        segment,
        scales,
        wavelet,
        sampling_period=1/fs
    )

    wavelet_power = np.abs(coefficients) ** 2

    mean_wavelet_power = np.mean(wavelet_power)

    max_wavelet_power = np.max(wavelet_power)

    average_wavelet_power = np.mean(
        wavelet_power,
        axis=1
    )

    dominant_wavelet_frequency = actual_frequencies[
        np.argmax(average_wavelet_power)
    ]

    # 9. Collect all features
    features = {

        "Record": record_name,

        "Sampling_Frequency": fs,

        "Original_Samples": len(signal),

        "Clean_Duration_sec": len(clean_signal) / fs,

        "Mean": mean_value,

        "Variance": variance,

        "Standard_Deviation": std_value,

        "RMS": rms,

        "Peak_to_Peak": peak_to_peak,

        "ZCR": zcr,

        "Waveform_Length": waveform_length,

        "Total_Band_Power": total_power,

        "Mean_Frequency": mean_frequency,

        "Median_Frequency": median_frequency,

        "Peak_Frequency": peak_frequency,

        "Mean_Spectrogram_Power": mean_spectrogram_power,

        "Max_Spectrogram_Power": max_spectrogram_power,

        "Dominant_STFT_Frequency": dominant_stft_frequency,

        "Mean_Wavelet_Power": mean_wavelet_power,

        "Max_Wavelet_Power": max_wavelet_power,

        "Dominant_Wavelet_Frequency": dominant_wavelet_frequency
    }

    return features


print("Feature extraction function created successfully!")

Feature extraction function created successfully!


In [17]:
test_features = extract_features("tpehg1007")

test_feature_df = pd.DataFrame([test_features])

print("Feature extraction successful!")

display(test_feature_df.T)

Feature extraction successful!


,0
Record,tpehg1007
Sampling_Frequency,20
Original_Samples,35180
Clean_Duration_sec,1399.0
Mean,-0.000021
Variance,0.000778
Standard_Deviation,0.027885
RMS,0.027885
Peak_to_Peak,0.269093
ZCR,0.03835


In [18]:
print("Number of extracted features:", len(test_features))

print("\nFeature names:")
for feature in test_features:
    print(feature)

print("\nMissing values:")
print(test_feature_df.isnull().sum().sum())

Number of extracted features: 21

Feature names:
Record
Sampling_Frequency
Original_Samples
Clean_Duration_sec
Mean
Variance
Standard_Deviation
RMS
Peak_to_Peak
ZCR
Waveform_Length
Total_Band_Power
Mean_Frequency
Median_Frequency
Peak_Frequency
Mean_Spectrogram_Power
Max_Spectrogram_Power
Dominant_STFT_Frequency
Mean_Wavelet_Power
Max_Wavelet_Power
Dominant_Wavelet_Frequency

Missing values:
0


In [19]:
extract_features("tpehg1007")
extract_features("tpehg1021")
extract_features("tpehg1022")

{'Record': 'tpehg1022',
 'Sampling_Frequency': 20,
 'Original_Samples': 35240,
 'Clean_Duration_sec': 1402.0,
 'Mean': np.float64(-2.3019137882677174e-06),
 'Variance': np.float64(0.0006837387692613274),
 'Standard_Deviation': np.float64(0.026148398980842546),
 'RMS': np.float64(0.026148399082164372),
 'Peak_to_Peak': np.float64(0.3188372625314717),
 'ZCR': np.float64(0.043403830379114806),
 'Waveform_Length': np.float64(57.904783703364615),
 'Total_Band_Power': np.float64(0.0006464452951476448),
 'Mean_Frequency': np.float64(0.2359899392263562),
 'Median_Frequency': np.float64(0.16701287975446225),
 'Peak_Frequency': np.float64(0.107421875),
 'Mean_Spectrogram_Power': np.float64(0.00013526184158563293),
 'Max_Spectrogram_Power': np.float64(0.013062675293403483),
 'Dominant_STFT_Frequency': np.float64(0.2734375),
 'Mean_Wavelet_Power': np.float64(0.0004990316924234055),
 'Max_Wavelet_Power': np.float64(0.040740150393409204),
 'Dominant_Wavelet_Frequency': np.float64(0.12962025316455697